# Along-Track Cost Decomposition — Compute

Per-leg decomposed costs for the best route per representative case (16 routes).
Saves to `../results/024_best_routes_per_leg_costs.parquet`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import msgpack
from tqdm.auto import tqdm
import warnings

sys.path.insert(0, str(Path("..").resolve()))

from experiment_params import REPRESENTATIVE_CASES, FORCING_BASELINE
from load_results import load_results, filter_kinked_routes, get_journey_params_df, get_elite_df
from ship_routing.app.routing import RoutingResult
from ship_routing.core.config import SHIP_DEFAULT, PHYSICS_DEFAULT
from ship_routing.core.cost import power_maintain_speed_decomposed
from ship_routing.core.data import load_currents, load_waves, load_winds, select_data_for_leg
from ship_routing.core.routes import Route, WayPoint

warnings.filterwarnings("ignore")

## Identify best route per representative case

In [2]:
# Load results and filter kinks
result_files = sorted(Path("../results").glob("cost_sub*.msgpack"))
rr_dict = load_results(result_files)
rr_dict = filter_kinked_routes(rr_dict)

journey_df = get_journey_params_df(rr_dict)
elite_df = get_elite_df(rr_dict)
elite_best = elite_df[elite_df.n_elite == 0].copy()
df = journey_df.join(elite_best)

df["month"] = pd.to_datetime(df.journey_time_start.astype(str)).dt.month
df["direction"] = df.journey_name.map(
    {"Atlantic_forward": "eastward", "Atlantic_backward": "westward"}
)
df["speed"] = df.journey_speed_knots.astype(float)

# Select best (lowest cost) per representative case
best_keys = {}
for ci, case in enumerate(REPRESENTATIVE_CASES):
    mask = (
        (df.journey_name == case["route"])
        & (df.month == case["month"])
        & (df.speed == case["speed"])
    )
    sub = df[mask].sort_values("elite_cost_absolute")
    best_keys[ci] = sub.index[0]
    print(f"Case {ci}: {case['route']} month={case['month']} {case['speed']}kn -> {sub.iloc[0].elite_cost_absolute/1e12:.3f} TJ")

print(f"\n{len(best_keys)} best routes identified")

records:   0%|          | 0/1052 [00:00<?, ?it/s]

135/1052 routes removed (max turning angle > 60.0°)


elite:   0%|          | 0/917 [00:00<?, ?it/s]

Case 0: Atlantic_forward month=8 8.0kn -> 1.935 TJ
Case 1: Atlantic_forward month=8 12.0kn -> 4.795 TJ
Case 2: Atlantic_forward month=8 16.0kn -> 7.118 TJ
Case 3: Atlantic_backward month=8 8.0kn -> 2.994 TJ
Case 4: Atlantic_backward month=8 12.0kn -> 5.066 TJ
Case 5: Atlantic_backward month=8 16.0kn -> 8.511 TJ
Case 6: Atlantic_forward month=1 8.0kn -> 8.439 TJ
Case 7: Atlantic_forward month=1 12.0kn -> 11.654 TJ
Case 8: Atlantic_forward month=1 16.0kn -> 13.284 TJ
Case 9: Atlantic_backward month=1 8.0kn -> 6.709 TJ
Case 10: Atlantic_backward month=1 12.0kn -> 9.160 TJ
Case 11: Atlantic_backward month=1 16.0kn -> 13.449 TJ

12 best routes identified


## Load forcing data

In [3]:
spatial_bounds = (-85, -5, 25, 60)
data_prefix = Path("..")

# Only load Jan + Aug (the months of the 16 representative cases)
datasets = {}
for month, tstart, tend in [(1, "2021-01-01", "2021-02-01"), (8, "2021-08-01", "2021-09-01")]:
    print(f"Loading month {month}...")
    datasets[month] = {
        "currents": load_currents(
            data_prefix / FORCING_BASELINE.currents_path,
            time_start=np.datetime64(tstart), time_end=np.datetime64(tend),
            engine=FORCING_BASELINE.engine, spatial_bounds=spatial_bounds,
            load_eagerly=True,
        ),
        "waves": load_waves(
            data_prefix / FORCING_BASELINE.waves_path,
            time_start=np.datetime64(tstart), time_end=np.datetime64(tend),
            engine=FORCING_BASELINE.engine, spatial_bounds=spatial_bounds,
            load_eagerly=True,
        ),
        "winds": load_winds(
            data_prefix / FORCING_BASELINE.winds_path,
            time_start=np.datetime64(tstart), time_end=np.datetime64(tend),
            engine=FORCING_BASELINE.engine, spatial_bounds=spatial_bounds,
            load_eagerly=True,
        ),
    }
print("Done.")

Loading month 1...


Loading month 8...


Done.


## Compute per-leg decomposed costs

In [4]:
def route_from_routing_result(rr, elite_idx=0):
    """Extract a Route with ns-precision timestamps from a RoutingResult."""
    route = rr.elite_population.members[elite_idx].route
    fixed_waypoints = [
        WayPoint(
            lon=wp.lon, lat=wp.lat,
            time=np.datetime64(wp.time, "ns"),
        )
        for wp in route.way_points
    ]
    return Route(way_points=tuple(fixed_waypoints))


def per_leg_decomposition(route, currents, waves, winds):
    """Compute per-leg decomposed costs with leg midpoint coordinates."""
    records = []
    cum_distance = 0.0
    cum_time_s = 0.0
    for leg in route.legs:
        costs = leg.cost_through_decomposed(
            current_data_set=currents,
            wind_data_set=winds,
            wave_data_set=waves,
            ship=SHIP_DEFAULT,
            physics=PHYSICS_DEFAULT,
        )

        # Compute wind no-current effect ourselves
        u_ship_og, v_ship_og = leg.uv_over_ground_ms
        ds_wind = select_data_for_leg(
            ds=winds,
            lon_start=leg.way_point_start.lon, lat_start=leg.way_point_start.lat,
            time_start=leg.way_point_start.time,
            lon_end=leg.way_point_end.lon, lat_end=leg.way_point_end.lat,
            time_end=leg.way_point_end.time,
        )
        ds_wave = select_data_for_leg(
            ds=waves,
            lon_start=leg.way_point_start.lon, lat_start=leg.way_point_start.lat,
            time_start=leg.way_point_start.time,
            lon_end=leg.way_point_end.lon, lat_end=leg.way_point_end.lat,
            time_end=leg.way_point_end.time,
        )
        _, _, pwr_wind_nc = power_maintain_speed_decomposed(
            u_current_ms=0, v_current_ms=0,
            u_wind_ms=ds_wind.uw, v_wind_ms=ds_wind.vw,
            w_wave_height=ds_wave.wh,
            u_ship_og_ms=u_ship_og, v_ship_og_ms=v_ship_og,
            ship=SHIP_DEFAULT, physics=PHYSICS_DEFAULT,
        )
        dt = leg.duration_seconds
        cost_wind_nc = pwr_wind_nc.mean().data[()] * dt

        costs["cost_wind_no_current"] = cost_wind_nc
        costs["delta_current_on_calm"] = costs["cost_calm_no_current"] - costs["cost_calm"]
        costs["delta_current_on_waves"] = costs["cost_waves_no_current"] - costs["cost_waves"]
        costs["delta_current_on_wind"] = cost_wind_nc - costs["cost_wind"]
        costs["delta_current_total"] = (
            costs["delta_current_on_calm"]
            + costs["delta_current_on_waves"]
            + costs["delta_current_on_wind"]
        )

        mid_lon = (leg.way_point_start.lon + leg.way_point_end.lon) / 2
        mid_lat = (leg.way_point_start.lat + leg.way_point_end.lat) / 2
        leg_len = leg.length_meters
        cum_distance += leg_len
        dt_s = (leg.way_point_end.time - leg.way_point_start.time) / np.timedelta64(1, "s")
        cum_time_s += dt_s

        costs["mid_lon"] = mid_lon
        costs["mid_lat"] = mid_lat
        costs["leg_length_m"] = leg_len
        costs["cum_distance_km"] = cum_distance / 1e3
        costs["duration_s"] = dt_s
        costs["cum_time_h"] = (cum_time_s - dt_s / 2) / 3600
        records.append(costs)
    return pd.DataFrame(records)

In [5]:
all_leg_data = []

for ci, key in tqdm(best_keys.items(), desc="routes"):
    rr = rr_dict[key]
    route = route_from_routing_result(rr)

    case = REPRESENTATIVE_CASES[ci]
    month = case["month"]
    ds = datasets[month]
    df_legs = per_leg_decomposition(route, ds["currents"], ds["waves"], ds["winds"])

    direction = "eastward" if "forward" in case["route"] else "westward"
    df_legs["month"] = month
    df_legs["direction"] = direction
    df_legs["speed"] = case["speed"]
    df_legs["case_idx"] = ci
    all_leg_data.append(df_legs)

df_all_legs = pd.concat(all_leg_data, ignore_index=True)
print(f"Total legs: {len(df_all_legs)} across {len(all_leg_data)} routes")

routes:   0%|          | 0/12 [00:00<?, ?it/s]

Total legs: 630 across 12 routes


## Save

In [6]:
Path("../results").mkdir(exist_ok=True)
df_all_legs.to_parquet("../results/024_best_routes_per_leg_costs.parquet", index=False)
print(f"Saved {len(df_all_legs)} leg records")

Saved 630 leg records
